# 03 - MLP with MLflow tracking
MLP (187 -> 128 -> 64 -> 5) on MIT-BIH heartbeats, class-weighted loss, early stopping on validation macro-F1.
Uses the same 80/20 stratified split as the Random Forest baseline. `mitbih_test.csv` is not used here.

In [ ]:
import copy
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import mlflow.pytorch
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
torch.manual_seed(SEED)

CLASS_NAMES = ["Normal", "Supraventricular", "Ventricular", "Fusion", "Unknown"]
config = {
    "batch_size": 256,
    "lr": 1e-3,
    "dropout": 0.3,
    "hidden_sizes": "128-64",
    "max_epochs": 50,
    "patience": 10,
    "weight_scheme": "balanced",  # "balanced" or "sqrt" (square root of balanced, a softer correction)
    "val_size": 0.2,
    "seed": SEED,
}

## Load data
No header. Columns 0-186 are the signal, column 187 is the label.

In [ ]:
df = pd.read_csv("../data/mitbih_train.csv", header=None)
X = df.iloc[:, :187].values             # (N, 187)
y = df.iloc[:, 187].values.astype(int)  # (N,)
print(X.shape, y.shape)
print("signal min/max:", X.min(), X.max())
print("label counts:", np.bincount(y))

## Validation split (same as the Random Forest baseline)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=config["val_size"], stratify=y, random_state=SEED
)
print(X_train.shape, X_val.shape)

## Tensors and DataLoaders
Features are float32; labels are int64 (required by `CrossEntropyLoss`).

In [ ]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)  # (N_train, 187)
y_train_t = torch.tensor(y_train, dtype=torch.long)     # (N_train,)
X_val_t = torch.tensor(X_val, dtype=torch.float32)      # (N_val, 187)
y_val_t = torch.tensor(y_val, dtype=torch.long)         # (N_val,)

train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t), batch_size=config["batch_size"], shuffle=True
)
val_loader = DataLoader(
    TensorDataset(X_val_t, y_val_t), batch_size=config["batch_size"], shuffle=False
)

xb, yb = next(iter(train_loader))
print(xb.shape, yb.shape)  # (256, 187), (256,)

## Model

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_features=187, n_classes=5, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 128),  # (B, 187) -> (B, 128)
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),           # (B, 128) -> (B, 64)
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, n_classes),     # (B, 64)  -> (B, 5) raw logits
        )

    def forward(self, x):
        return self.net(x)


model = MLP(dropout=config["dropout"])
print(model)
print("logits shape:", model(xb).shape)  # (256, 5)

## Loss and optimizer
Class weights from the **training** labels only. `balanced`: `n_samples / (n_classes * class_count)`; `sqrt`: the square root of that (softer correction). Set `weight_scheme` in the config cell.

In [ ]:
class_counts = np.bincount(y_train, minlength=5)
class_weights = len(y_train) / (5 * class_counts)  # balanced
if config["weight_scheme"] == "sqrt":
    class_weights = np.sqrt(class_weights)
elif config["weight_scheme"] != "balanced":
    raise ValueError(f"unknown weight_scheme: {config['weight_scheme']}")
print("weight scheme:", config["weight_scheme"])
print("class weights:", dict(zip(CLASS_NAMES, class_weights.round(3))))

criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32))
optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])

## Train and eval steps

In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:                 # xb (B, 187), yb (B,)
        optimizer.zero_grad()
        logits = model(xb)                # (B, 5)
        loss = criterion(logits, yb)      # scalar
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    for xb, yb in loader:
        logits = model(xb)                # (B, 5)
        total_loss += criterion(logits, yb).item() * xb.size(0)
        all_preds.append(logits.argmax(dim=1))  # (B,)
        all_labels.append(yb)
    preds = torch.cat(all_preds).numpy()      # (N_val,)
    labels = torch.cat(all_labels).numpy()    # (N_val,)
    macro_f1 = f1_score(labels, preds, average="macro")
    return total_loss / len(loader.dataset), macro_f1, preds, labels

## Training with MLflow
Early stopping on val macro-F1 (patience 10); best weights are restored at the end.

In [ ]:
# MLflow 3.x no longer allows the plain ./mlruns file store, so use SQLite (kept at the project root)
mlflow.set_tracking_uri(f"sqlite:///{(Path.cwd().parent / 'mlflow.db').as_posix()}")
mlflow.set_experiment("ecg-arrhythmia")

history = {"train_loss": [], "val_loss": [], "val_f1": []}
best_f1, best_state, epochs_no_improve = -1.0, None, 0

with mlflow.start_run(run_name=f"mlp-{config['weight_scheme']}"):
    mlflow.log_params(config)

    for epoch in range(1, config["max_epochs"] + 1):
        train_loss = train_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_f1, _, _ = eval_epoch(model, val_loader, criterion)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_f1"].append(val_f1)
        mlflow.log_metrics(
            {"train_loss": train_loss, "val_loss": val_loss, "val_macro_f1": val_f1},
            step=epoch,
        )
        print(f"epoch {epoch:2d} | train {train_loss:.4f} | val {val_loss:.4f} | val macro-F1 {val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= config["patience"]:
                print(f"early stopping at epoch {epoch} (best val macro-F1 {best_f1:.4f})")
                break

    model.load_state_dict(best_state)
    _, final_f1, val_preds, val_labels = eval_epoch(model, val_loader, criterion)
    report = classification_report(val_labels, val_preds, target_names=CLASS_NAMES, digits=4)

    mlflow.log_metric("final_val_macro_f1", final_f1)
    mlflow.log_text(report, "classification_report.txt")
    mlflow.pytorch.log_model(
        model,
        name="model",
        input_example=X_val[:5].astype(np.float32),  # (5, 187), records the input signature
        serialization_format="pickle",
    )

## Results

In [ ]:
print(report)
print("Validation macro-F1:", round(final_f1, 4))

In [ ]:
cm = confusion_matrix(val_labels, val_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Validation confusion matrix")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history["train_loss"], label="train loss")
plt.plot(history["val_loss"], label="val loss")
plt.xlabel("Epoch")
plt.ylabel("Weighted cross-entropy")
plt.title("Loss curves")
plt.legend()
plt.tight_layout()
plt.show()